<a href="https://colab.research.google.com/github/Kmjng/BodyFat-Predict/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A6%9D%EA%B0%95_%EC%8B%9C%EB%8F%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
Created on Thu May 16 18:03:43 2024
@author: itwill
"""
import pandas as pd
file_2 = r"/content/drive/MyDrive/Colab Notebooks/중간프로젝트/Dataset.csv"

df_f = pd.read_csv(file_2, encoding= 'euc-kr')
df_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  177 non-null    int64  
 1   BodyFat     177 non-null    float64
 2   Original    177 non-null    object 
 3   Sex         177 non-null    object 
 4   Age         177 non-null    int64  
 5   Weight      177 non-null    float64
 6   Height      177 non-null    float64
 7   Neck        177 non-null    float64
 8   Chest       177 non-null    float64
 9   Abdomen     177 non-null    float64
 10  Hip         177 non-null    float64
 11  Thigh       177 non-null    float64
 12  Knee        177 non-null    float64
 13  Ankle       177 non-null    float64
 14  Biceps      177 non-null    float64
 15  Forearm     177 non-null    float64
 16  Wrist       177 non-null    float64
 17  BMI         177 non-null    float64
 18  AC_ratio    177 non-null    float64
 19  HT_ratio    177 non-null    f

In [ ]:
df_f = df_f[['Height','Weight','Chest','Abdomen','Hip','Thigh','Biceps','Ankle','Knee','Neck','Wrist','BodyFat','Class']]
# # 10~27, 28~37

In [ ]:
df_f[['BodyFat','Class']]

,BodyFat,Class
0,23.07,0
1,29.50,1
2,26.99,0
3,20.25,0
4,19.95,0
...,...,...
172,15.83,0
173,30.40,1
174,11.98,0
175,11.24,0


In [ ]:
df_f.BodyFat.min()

10.07

In [ ]:
#df_f['Class'] = df_f['Class'].astype(int)

from collections import Counter
cnt = Counter(df_f['Class'])
cnt # Counter({0: 157, 1: 20})

Counter({0: 157, 1: 20})

In [ ]:
len(df_f.columns) # 13
df_f.columns

Index(['Height', 'Weight', 'Chest', 'Abdomen', 'Hip', 'Thigh', 'Biceps',
       'Ankle', 'Knee', 'Neck', 'Wrist', 'BodyFat', 'Class'],
      dtype='object')

In [ ]:
###############
#### 연속형 변수만 포함
###############

df_f_no_class = df_f.drop(['Class'],axis = 1)
len(df_f_no_class.columns)

12

In [ ]:

##########################################
#### 고유값분해를 이용한 데이터 증강 알고리즘
##########################################

# BodyFat 분산 확인 (표준화 전)
print(df_f.BodyFat.var() ,'/',df_f.BodyFat.mean() )

30.361087249614794 / 21.717457627118645


In [ ]:
import numpy as np


###############
###데이터 MinMaxScaler
###############
from sklearn.preprocessing import MinMaxScaler, StandardScaler

def standard_df(df):
    scaler = StandardScaler()
    normalized_values = scaler.fit_transform(df.values)
    standarded_df = pd.DataFrame(normalized_values, columns=df.columns, index=df.index)
    return standarded_df, scaler

def inverse_standard_df(standarded_df, scaler):

    original_values = scaler.inverse_transform(standarded_df.values)
    original_df = pd.DataFrame(original_values, columns=standarded_df.columns, index=standarded_df.index)
    return original_df

'''
def minmax_df(df):
    scaler = MinMaxScaler()
    normalized_values = scaler.fit_transform(df.values)
    normalized_df = pd.DataFrame(normalized_values, columns=df.columns, index=df.index)
    return normalized_df, scaler

def inverse_minmax_df(normalized_df, scaler):

    original_values = scaler.inverse_transform(normalized_df.values)
    original_df = pd.DataFrame(original_values, columns=normalized_df.columns, index=normalized_df.index)
    return original_df
'''

'\ndef minmax_df(df):\n    scaler = MinMaxScaler()\n    normalized_values = scaler.fit_transform(df.values)\n    normalized_df = pd.DataFrame(normalized_values, columns=df.columns, index=df.index)\n    return normalized_df, scaler\n\ndef inverse_minmax_df(normalized_df, scaler):\n\n    original_values = scaler.inverse_transform(normalized_df.values)\n    original_df = pd.DataFrame(original_values, columns=normalized_df.columns, index=normalized_df.index)\n    return original_df\n'

In [ ]:
# minmax_m, scaler = minmax_df(df_m_no_class)
standarded_df, scaler = standard_df(df_f_no_class)

# BodyFat에 대한 표준화 확인
print("최소값: ", standarded_df.BodyFat.min(),'/평균값: ', standarded_df.BodyFat.mean(),'/표준편차: ', standarded_df.BodyFat.std())


최소값:  -2.1198385150439982 /평균값:  -1.2043097216272884e-16 /표준편차:  1.0028368851322822


In [ ]:
def calculate_covariance_matrix(x):
    cov_matrix = np.cov(x, rowvar=False)
    return cov_matrix

def calculate_eigen(cov_matrix):
    # 주성분 계산
    eigen_values, eigen_vectors = np.linalg.eig(cov_matrix)
    # 내림차순으로 정렬
    idx = np.argsort(eigen_values)[::-1]
    eigen_values = eigen_values[idx]
    eigen_vectors = eigen_vectors[:, idx]
    return eigen_values, eigen_vectors, idx

def split_components(eigen_vectors, d):
    U1 = eigen_vectors[:, :d]  # 주성분(내림차순이니까)
    U2 = eigen_vectors[:, d:]  # 비주성분
    return U1, U2

def project_data(x, U1, U2):
    q = np.dot(x , U1) # (250,19) * (19,10)
    s = np.dot(x , U2)
    return q, s

def sample_q_prime(q_mean, q_variance, size):
    q_prime = np.random.normal(q_mean, np.sqrt(q_variance), size)
    # np.random.normal(평균값, 표준편차, 생성할 난수 갯수)
    # 정규분포를 따르는 난수 생성 함수
    # size=(250,10) 이면, 10개 열에 대한 난수 생성임.
    return q_prime

# generate_sample에서 난수로 생성된 q_prime 과 s를 합침
def generate_sample(q_prime, s, U1, U2):
    sample = np.dot(q_prime, U1.T) + np.dot(s, U2.T)
    return sample # (250,19)

def inverse_projection(sample, U):
    x_prime = np.dot(sample , U.T)
    return x_prime

def augment_data(x, x_prime):
    x_aug = np.vstack((x, x_prime))
    return x_aug


In [ ]:
# 랜덤 시드 설정
np.random.seed(40)

###########
# 1. 데이터 생성

#x = minmax_m.values
x = standarded_df.values


# 2. 공분산 행렬 계산 (주대각선 1에 가까움)
cov_matrix_of_x = calculate_covariance_matrix(x)
cov_matrix_of_x.shape # 13,13


# 4. 고유값과 고유벡터 계산
eigen_values, eigen_vectors, idx = calculate_eigen(cov_matrix_of_x)

'''
주성분/ 비주성분 내림차순으로 배열됨
이상한 점 : 내림차순 전/후가 거의 같음 => 기존 열 순서가 주성분 순서라는..
'''
idx # array([ 0,  1,  2,  3,  8, 10, 11,  9,  7,  6,  5,  4])

array([ 0,  1,  2,  3,  8, 10, 11,  9,  7,  6,  5,  4])

In [ ]:
# 5. 주성분과 비주성분 분리
d = 8  # 주성분 개수
U1, U2 = split_components(eigen_vectors, d)
U1.shape # 12,8
U2.shape # 12,4

# 6. 주성분과 비주성분에 데이터 사영
q, s = project_data(x, U1, U2)

# 7. q'를 샘플링
q_mean = np.mean(q, axis=0)
q_mean # 10개 주성분 칼럼에 대한 평균
q_variance = np.var(q, axis=0)
q_variance # 10개 주성분 칼럼에 대한 분산
q_prime = sample_q_prime(q_mean, q_variance, size=(df_f.shape[0], d))

# 8. 샘플링된 q'와 s를 단순히 합침
sample = generate_sample(q_prime, s, U1, U2)
U1.shape # (12, 8)
U2.shape # (12, 4)
s.shape # (177, 4)
q_prime.shape # (177, 8)

# 9. 역사영하여 샘플링 데이터 x' 생성
x_prime = inverse_projection(sample, eigen_vectors)
x_prime.shape # (177, 12)


(177, 12)

In [ ]:
# 정규성 난수로 생성된 데이터 x_prime 기초통계량 확인
# BodyFat 0번째열
print(f'최소:{x_prime[0].min()}/평균:{x_prime[0].mean()}/표준편차:{x_prime[0].std()}')


최소:-1.1601040368854496/평균:-0.055104293501742424/표준편차:0.7622881533768158


In [ ]:
# 10. 증강된 데이터 x_aug 생성(vstack)
x_aug = augment_data(x, x_prime)

x_aug.shape #  (354, 12)

(354, 12)

In [ ]:
# 결과 확인
print("원본 데이터 x의 형태:", x.shape)
print("데이터 증강 후 x_aug의 형태:", x_aug.shape)

원본 데이터 x의 형태: (177, 12)
데이터 증강 후 x_aug의 형태: (354, 12)


In [ ]:
df_f_no_class.columns
x_aug = pd.DataFrame(x_aug, columns = df_f_no_class.columns)

#aug_m = inverse_minmax_df(x_aug, scaler)


# 역 표준화
aug_m = inverse_standard_df(x_aug, scaler)
aug_m.shape # (358, 12)
aug_m.BodyFat.min() # 8.061289282982916
aug_m.describe().T


,count,mean,std,min,25%,50%,75%,max
Height,354.0,167.357442,8.139837,141.262351,162.086197,166.785527,172.700000,191.191824
Weight,354.0,60.218145,7.558454,41.597165,54.762719,59.900000,65.300000,78.499975
Chest,354.0,85.780149,4.711965,70.438764,82.936781,85.584791,88.694215,100.000000
Abdomen,354.0,69.284234,5.832660,52.791565,65.000000,68.363274,72.803951,96.000000
Hip,354.0,96.931020,5.404674,84.000000,93.077102,96.707660,100.500000,115.000000
Thigh,354.0,51.480514,4.723424,38.000000,48.499081,51.481084,54.500000,65.501009
Biceps,354.0,26.693149,2.061307,21.000000,25.500000,26.734691,28.000000,33.000000
Ankle,354.0,21.280888,1.312233,17.329765,20.344967,21.291144,22.127293,25.000000
Knee,354.0,35.755006,1.844461,24.700000,34.500000,35.700000,36.884549,42.000000
Neck,354.0,31.526847,1.463366,26.000000,30.500000,31.500000,32.528145,36.000000


In [ ]:
aug_m.BodyFat.min()

8.061289282982916

In [ ]:
# 생성된 데이터 중 bodyFat 10 미만(이상치 기준)인 데이터 제외
aug_m = aug_m[aug_m['BodyFat'] > 10]
aug_m.shape # (350, 13) # # 생성된 데이터 5개가 이상치

(353, 12)

In [ ]:
aug_m.to_csv(r"/content/drive/MyDrive/Colab Notebooks/중간프로젝트/Oversampling_eigenvalue_decomposition_Female.csv", index = False)

In [ ]:
##############################
######증강된 데이터 클래스 분류
##############################

# 이진 클래스 라벨링
def class_2_labeling(df):
    class_lst = [0,1]
    # class 0 ; 비만 아님
    # class 1 ;  비만임

    df['Class'] = None
    for i, values in df.iterrows():
      if df['BodyFat'][i] < 24 : # 여성 30세 미만 기준
        df['Class'][i] = class_lst[0]
      elif  df['BodyFat'][i] >= 24 :
        df['Class'][i] = class_lst[1]


    return df
aug_m = class_2_labeling(aug_m) # 라벨링

<ipython-input-69-7489ec04852d>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Class'][i] = class_lst[0]
<ipython-input-69-7489ec04852d>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Class'][i] = class_lst[1]
<ipython-input-69-7489ec04852d>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Class'][i] = class_lst[1]
<ipython-input-69-7489ec04852d>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

In [ ]:
aug_m.info()

<class 'pandas.core.frame.DataFrame'>
Index: 353 entries, 0 to 353
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Height   353 non-null    float64
 1   Weight   353 non-null    float64
 2   Chest    353 non-null    float64
 3   Abdomen  353 non-null    float64
 4   Hip      353 non-null    float64
 5   Thigh    353 non-null    float64
 6   Biceps   353 non-null    float64
 7   Ankle    353 non-null    float64
 8   Knee     353 non-null    float64
 9   Neck     353 non-null    float64
 10  Wrist    353 non-null    float64
 11  BodyFat  353 non-null    float64
 12  Class    353 non-null    object 
dtypes: float64(12), object(1)
memory usage: 46.7+ KB


In [ ]:
cnt = Counter(aug_m['Class'])
cnt

Counter({0: 231, 1: 122})

In [ ]:
# 라벨링 후 덮어쓰기
aug_m.to_csv(r"/content/drive/MyDrive/Colab Notebooks/중간프로젝트/Oversampling_eigenvalue_decomposition_Female.csv", index = False)
